# Granger causality test

In [1]:
import os
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests

In [2]:
REPO_ROOT = "/Users/beszabo/bene/szakdolgozat"
DERIVED_DIR = os.path.join(REPO_ROOT, "data", "panels")

# Use enriched panel (no pre-computed lags) for Granger causality
PANEL_CSV = os.path.join(DERIVED_DIR, "company_weekly_panel_enriched.csv")

panel = pd.read_csv(PANEL_CSV, parse_dates=['week_start'])


In [ ]:
# Aggregate to time series (sum across all companies per week)
ts_data = panel.groupby('week_start').agg({
    'num_memes': 'sum',
    'NYT_mention': 'sum'
}).sort_index()

print("Null hypothesis: NYT mentions do NOT Granger-cause meme activity")
print("If p < 0.05, we reject the null → NYT mentions help predict future memes\n")

# Test: Does NYT_mention → num_memes?
result_nyt_to_memes = grangercausalitytests(
    ts_data[['num_memes', 'NYT_mention']], 
    maxlag=4,
    verbose=True
)

print("Null hypothesis: Meme activity does NOT Granger-cause NYT mentions")
print("If p < 0.05, we reject the null → Memes help predict future NYT coverage\n")

# Test: Does num_memes → NYT_mention?
result_memes_to_nyt = grangercausalitytests(
    ts_data[['NYT_mention', 'num_memes']], 
    maxlag=4,
    verbose=True
)


GRANGER CAUSALITY TEST: Does NYT mention → meme activity?
Null hypothesis: NYT mentions do NOT Granger-cause meme activity
If p < 0.05, we reject the null → NYT mentions help predict future memes


Granger Causality
number of lags (no zero) 1
ssr based F test:         F=3.2194  , p=0.0757  , df_denom=102, df_num=1
ssr based chi2 test:   chi2=3.3141  , p=0.0687  , df=1
likelihood ratio test: chi2=3.2628  , p=0.0709  , df=1
parameter F test:         F=3.2194  , p=0.0757  , df_denom=102, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=1.1814  , p=0.3111  , df_denom=99, df_num=2
ssr based chi2 test:   chi2=2.4822  , p=0.2891  , df=2
likelihood ratio test: chi2=2.4530  , p=0.2933  , df=2
parameter F test:         F=1.1814  , p=0.3111  , df_denom=99, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=1.1435  , p=0.3356  , df_denom=96, df_num=3
ssr based chi2 test:   chi2=3.6806  , p=0.2981  , df=3
likelihood ratio test: chi2=

/Users/beszabo/bene/szakdolgozat/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/Users/beszabo/bene/szakdolgozat/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


In [4]:
# Extract and display p-values for easier interpretation
print("\n" + "=" * 80)
print("SUMMARY OF RESULTS")
print("=" * 80)

print("\n1. NYT mentions → Meme activity:")
for lag in range(1, 5):
    # F-test p-value from ssr_ftest
    p_val = result_nyt_to_memes[lag][0]['ssr_ftest'][1]
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
    print(f"   Lag {lag}: p = {p_val:.4f} {sig}")

print("\n2. Meme activity → NYT mentions:")
for lag in range(1, 5):
    p_val = result_memes_to_nyt[lag][0]['ssr_ftest'][1]
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
    print(f"   Lag {lag}: p = {p_val:.4f} {sig}")

print("\n" + "-" * 80)
print("Interpretation:")
print("  * p < 0.05: Reject null hypothesis (Granger causality detected)")
print("  *** p < 0.001, ** p < 0.01, * p < 0.05, n.s. = not significant")
print("=" * 80)



SUMMARY OF RESULTS

1. NYT mentions → Meme activity:
   Lag 1: p = 0.0757 n.s.
   Lag 2: p = 0.3111 n.s.
   Lag 3: p = 0.3356 n.s.
   Lag 4: p = 0.4307 n.s.

2. Meme activity → NYT mentions:
   Lag 1: p = 0.0167 *
   Lag 2: p = 0.7012 n.s.
   Lag 3: p = 0.9068 n.s.
   Lag 4: p = 0.9890 n.s.

--------------------------------------------------------------------------------
Interpretation:
  * p < 0.05: Reject null hypothesis (Granger causality detected)
  *** p < 0.001, ** p < 0.01, * p < 0.05, n.s. = not significant
